# Theory guess generation and experimental fitting of UV-Vis spectra

Experimental UV spectra is made of different bands that can have different line shapes. It is possible to generate a theoretical guess using semi-empirical or ab-initio calculation and fit the guess to experimental results.

## Setting up the calculations

First thing we need to import the necessary modules.

In [ ]:
from pathlib import Path

from pydftb.io.parser import parse_xyzfile
from vibrofit.plotting.summaries import plot_comprehensive_summary
from vibrofit.utils.printing import print_mixed_results, print_parameter_comparison
from vibrofit.workflows import compute_multistate_spectrum

from hqs_uv_vis.api import AutoVibronicFitter
from hqs_uv_vis.orchestrator import QuantumOrchestrator
from hqs_uv_vis.schema import default_angular_momentum, MoleculeStructure, WorkflowConfig

In [ ]:
# Path to the stored data
DATA_DIR = Path.cwd()

Load the cartesian coordinates of the molecule you are interested in.

In [ ]:
xyz_path = DATA_DIR / "phthalimide.xyz"
xyz_data = parse_xyzfile(xyz_path)
molecule = MoleculeStructure(
    symbols=xyz_data["symbols"],
    coordinates=xyz_data["coordinates"],
)

Configure the calculations. The most important configs are mentioned below:

- band_types: select the number and the shape of bands you want to be simulated.

- sk_prefix: DFTB calculations require Slater-Koster parameters you can download them from https://dftb.org/parameters.html

- num_excitations: Number of excited states to be calculated for initial guess.


In [ ]:
PATH_TO_SK_PARAMS = "/path-to-params-folder/3ob-3-1/" # WARNING! you need to adapt the path
config = WorkflowConfig(
    band_types=["pekar", "pekar", "gaussian"],
    peak_prominence=0.05,
    sk_prefix=PATH_TO_SK_PARAMS,
    angular_momentum=default_angular_momentum(parse_xyzfile(xyz_path)["symbols"]),
    num_excitations=10,
    wavelength_range_nm=(200.0, 800.0),
    min_osc_strength=0.05,
    run_bootstrap=False,
)

## Running calculations and displaying the results

If you like, you can only calculate the theoretical spectrum. This separated step is not necessary if you are interested in fitting the spectrum.

In [ ]:
# Setting up theoretical calculations
orchestrator = QuantumOrchestrator(config)

# Running theoretical calculations with TD-DFTB
data = orchestrator.run_pipeline(molecule, debug=True)

# Generate the theoretical spectrum from the raw quantum outputs
theory_results = compute_multistate_spectrum(
    transitions=data["transitions"],
    omega_au=data["omega_au"],
    L=data["L"],
    M_au=data["M_au"]
)

For fitting to the experimental spectrum you only need to load your molecule, the experimental data and execute the `AutoVibronicFitter` class.

In [ ]:
exp_spectrum_path = DATA_DIR / "phthalimide_exp.npy"

fitter = AutoVibronicFitter(config)
result = fitter.execute(molecule=molecule, exp_file=exp_spectrum_path, debug=True)

You can use the extended print and plotting functions to analyze the results.

In [ ]:
print("\nFitting parameters for each band:")
print_mixed_results(
        result.optimized_parameters, result.parameter_covariance, config.band_types
)
print(
    "Deviation between theoretic and fitted parameters for the "
    f"{result.bright_states_analyzed} bright state(s)/band(s) found theoretically"
)
print_parameter_comparison(result.optimized_parameters, result.theory_results)
plot_comprehensive_summary(
        result.exp_spectrum["frequencies_cm1"],
        result.exp_spectrum["intensities"],
        result.optimized_parameters,
        config.band_types,
        theory_results=result.theory_results,
        output_path=DATA_DIR / f"phthalimide_summary_{result.strategy_used}.png",
    )